In [2]:
import torch
import math
from ot.batch import solve_gromov_batch

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 0) 네 로그 분포(대충) 타겟
#   mean~24042, q50~12553, q90~62694 (예시)
#   여기서는 LogNormal로 "모양"만 맞춤
# =========================
B, M, N, K = 4, 16, 8, 8
D = 768
alpha = 0.5
reg = 0.05
soft_tau = 0.05

target_mean = 24042.0
target_q50  = 12552.9
target_q90  = 62693.6

# LogNormal 파라미터(대충 맞추기):
# median = exp(mu) = q50  => mu = log(q50)
# q90 = exp(mu + sigma*z0.90) => sigma = (log(q90)-mu)/z0.90
z90 = 1.281551565545  # N(0,1) 90% quantile
mu = math.log(target_q50)
sigma = (math.log(target_q90) - mu) / z90

dist_sampler = torch.distributions.LogNormal(loc=torch.tensor(mu, device=device),
                                             scale=torch.tensor(sigma, device=device))

# dist_sq를 직접 샘플링: [B*M, N, K]
dist_sq = dist_sampler.sample((B*M, N, K)).clamp_min(1e-6)

# =========================
# 1) 구조(cost)도 그냥 랜덤 attention cost로 만듦 (0~1대 값)
#   - src_str: [B*M, N, N]
#   - tgt_str: [B*M, K, K]
# =========================
def rand_attn_cost(batch, size, proj_dim=32):
    X = torch.randn(batch, size, proj_dim, device=device)
    scores = (X @ X.transpose(-2, -1)) / math.sqrt(proj_dim)
    attn = torch.softmax(scores, dim=-1)
    return 1.0 - attn  # cost

src_str = rand_attn_cost(B*M, N)
tgt_str = rand_attn_cost(B*M, K)

# =========================
# 2) 분포 찍기
# =========================
@torch.no_grad()
def qprint(x, name):
    x = x.detach().float().flatten()
    qs = torch.quantile(x, torch.tensor([0.0,0.5,0.9,0.95,0.99,1.0], device=x.device))
    print(f"\n[{name}] mean={x.mean().item():.1f} std={x.std().item():.1f}")
    print(f"[{name}] q00={qs[0].item():.1f} q50={qs[1].item():.1f} q90={qs[2].item():.1f} "
          f"q95={qs[3].item():.1f} q99={qs[4].item():.1f} q100={qs[5].item():.1f}")

qprint(dist_sq, "dist_sq (sampled)")
dist_norm = dist_sq / float(D)
qprint(dist_norm, "dist_sq/D (=dist_norm)")

# =========================
# 3) FGW: RAW vs mean-scaling
# =========================
a = torch.ones(B*M, N, device=device) / N
b = torch.ones(B*M, K, device=device) / K

M_cost_raw = dist_norm

# mean-scaling (음수 없음, 정보 "단조" 유지)
sf = dist_norm.detach().mean().clamp_min(1e-8)
M_cost_scaled = dist_norm / sf

ss = src_str.detach().mean().clamp_min(1e-8)
st = tgt_str.detach().mean().clamp_min(1e-8)
src_str_scaled = src_str / ss
tgt_str_scaled = tgt_str / st

def run_fgw(src_s, tgt_s, M_cost, tag):
    out = solve_gromov_batch(
        src_s, tgt_s, M=M_cost, alpha=alpha, reg=reg, a=a, b=b,
        max_iter=10, tol=1e-3, grad="envelope"
    )
    T = out.plan.detach()
    val = out.value.detach()

    feat_term = (M_cost * T).sum(dim=(1,2)).mean().item()
    total_val = val.mean().item()
    struct_term = (total_val - (1 - alpha) * feat_term) / (alpha + 1e-9)
    ratio = feat_term / (struct_term + 1e-9)

    print(f"\n[{tag}]")
    print(f"  M_cost mean={M_cost.mean().item():.4f}, std={M_cost.std().item():.4f}")
    print(f"  src_str mean={src_s.mean().item():.4f}, tgt_str mean={tgt_s.mean().item():.4f}")
    print(f"  FGW value mean={total_val:.6f}")
    print(f"  Feat term={feat_term:.6f} | Struct term={struct_term:.6f} | Ratio={ratio:.3f}")
    return out

out_raw = run_fgw(src_str, tgt_str, M_cost_raw, "RAW")
out_scl = run_fgw(src_str_scaled, tgt_str_scaled, M_cost_scaled, "SCALED (CF+CS mean-scaling)")

# =========================
# 4) pi 체크 (샘플 단위 B로 reshape)
# =========================
d_raw = out_raw.value.reshape(B, M)
d_scl = out_scl.value.reshape(B, M)
pi_raw = torch.softmax(-d_raw / soft_tau, dim=1)
pi_scl = torch.softmax(-d_scl / soft_tau, dim=1)

print("\n[PI CHECK]")
print(" raw   pi[0]:", pi_raw[0].detach().cpu().numpy().round(4))
print(" scaled pi[0]:", pi_scl[0].detach().cpu().numpy().round(4))
print(" raw   max(pi)[0]:", float(pi_raw[0].max()))
print(" scaled max(pi)[0]:", float(pi_scl[0].max()))



[dist_sq (sampled)] mean=26445.2 std=45818.3
[dist_sq (sampled)] q00=247.1 q50=12323.5 q90=63270.6 q95=97294.4 q99=222641.9 q100=805115.9

[dist_sq/D (=dist_norm)] mean=34.4 std=59.7
[dist_sq/D (=dist_norm)] q00=0.3 q50=16.0 q90=82.4 q95=126.7 q99=289.9 q100=1048.3

[RAW]
  M_cost mean=34.4339, std=59.6592
  src_str mean=0.8750, tgt_str mean=0.8750
  FGW value mean=2.372643
  Feat term=4.718647 | Struct term=0.026640 | Ratio=177.127

[SCALED (CF+CS mean-scaling)]
  M_cost mean=1.0000, std=1.7326
  src_str mean=1.0000, tgt_str mean=1.0000
  FGW value mean=0.074412
  Feat term=0.146153 | Struct term=0.002672 | Ratio=54.703

[PI CHECK]
 raw   pi[0]: [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 scaled pi[0]: [0.0453 0.089  0.0432 0.1418 0.0766 0.0441 0.0534 0.0603 0.0387 0.0493
 0.075  0.0787 0.0614 0.048  0.0399 0.0552]
 raw   max(pi)[0]: 0.9999940395355225
 scaled max(pi)[0]: 0.14182579517364502


## Coordinate Clustering

- H_sample norm histogram (want left)
    - 각 샘플의 entropy H(pi)를 최대 엔트로피 log M으로 나누어 normalize
    - 0에 가까우면 거의 one hot이고 
    - 1에 가까우면 uniform 
    - 현재 막대가 0.98 ~ 1.0 에 몰려 있어서 sample entropy 낮게와는 정반대

- top1 prob histogram (want right)
    - 각 샘플에서 max_m pi_m 가장 큰 값의 분포 
    - flat이면 top 1이 1/M

- top1 - top2 histogram 
    - 각 샘플에서 (1등 확률 - 2등 확률) 차이 
    - 결정의 확신도 
    - 샤프하면 top1이 크고, top2가 작아서 margin이 크게 나옴 
    - 애매하면 top 1 ~ top 2여서 margin이 0 근처 
    - 현재 0 ~ 0.03 근처여서 top1과 top2가 거의 비슷함.

< 결과 >
- Top 1 과 Top 2의 차이가 거의 없다면 이들의 cost 차이가 없음을 의미 